# Run a training script as a command job

You can use the Python SDK for Azure Machine Learning to submit scripts as command jobs. By using jobs, you can easily keep track of the input parameters and outputs when training a machine learning model.

## Before you start

You'll need the latest version of the **azureml-ai-ml** package to run the code in this notebook. Run the cell below to verify that it is installed.

> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

In [1]:
pip show azure-ai-ml

Name: azure-ai-ml
Version: 1.32.0
Summary: Microsoft Azure Machine Learning Client Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python
Author: Microsoft Corporation
Author-email: azuresdkengsysadmins@microsoft.com
License: MIT License
Location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-mgmt-core, azure-monitor-opentelemetry, azure-storage-blob, azure-storage-file-datalake, azure-storage-file-share, colorama, isodate, jsonschema, marshmallow, pydash, pyjwt, pyyaml, strictyaml, tqdm, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## Connect to your workspace

With the required SDK packages installed, now you're ready to connect to your workspace.

To connect to a workspace, we need identifier parameters - a subscription ID, resource group name, and workspace name. Since you're working with a compute instance, managed by Azure Machine Learning, you can use the default values to connect to the workspace.

In [2]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [3]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


## Custom tracking with MLflow

When running a script as a job you can use MLflow in your training script to track the model. MLflow allows you to track any custom parameters, metrics, or artifacts you want to store with your job output.

Run the following cells to create the **train-model-mlflow.py** script in the **src** folder. The script trains a classification model by using the **diabetes.csv** file in the same folder, which is passed as an argument. 

Review the code below to find that the script will import `mlflow` and log:

- The regularization rate as a **parameter**. 
- The accuracy and AUC as **metrics**.
- The plotted ROC curve as an **artifact**.

In [4]:
import os

# create a folder for the script files
script_folder = 'src'
os.makedirs(script_folder, exist_ok=True)
print(script_folder, 'folder created')

src folder created


In [5]:
%%writefile $script_folder/train-model-mlflow.py

# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)

    # label encoding categorical columns
    le_arrival = LabelEncoder()
    le_symptom = LabelEncoder()
    le_target = LabelEncoder()

    df['Arrival_Mode'] = le_arrival.fit_transform(df['Arrival_Mode'])
    df['Primary_Symptom'] = le_symptom.fit_transform(df['Primary_Symptom'])
    df['Routing_Class'] = le_target.fit_transform(df['Routing_Class'])

    # normalization
    scaler = MinMaxScaler()

    df[['Age', 'Arrival_Mode', 'Heart_Rate',
        'Pain_Scale', 'Primary_Symptom']] = scaler.fit_transform(
        df[['Age', 'Arrival_Mode', 'Heart_Rate',
        'Pain_Scale', 'Primary_Symptom']]
    )

    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Age','Arrival_Mode','Heart_Rate','Pain_Scale',
    'Primary_Symptom']].values, df['Routing_Class'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=0
    )

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)

    print("Training model...")

    model = LogisticRegression(
        C=1/reg_rate,
        solver="liblinear"
    ).fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):

    # calculate accuracy
    y_hat = model.predict(X_test)

    acc = np.average(y_hat == y_test)

    print('Accuracy:', acc)

    mlflow.log_metric("Accuracy", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)

    auc = roc_auc_score(y_test, y_scores[:,1])

    print('AUC: ' + str(auc))

    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])

    fig = plt.figure(figsize=(6, 4))

    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')

    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')

    plt.savefig("ROC-Curve.png")

    mlflow.log_artifact("ROC-Curve.png")

def parse_args():

    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument(
        "--training_data",
        dest='training_data',
        type=str
    )

    parser.add_argument(
        "--reg_rate",
        dest='reg_rate',
        type=float,
        default=0.01
    )

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":

    # add space in logs
    print("\n\n")

    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)

    print("\n\n")

Writing src/train-model-mlflow.py


Now, you can submit the script as a command job.

Run the cell below to train the model. 

In [6]:
from azure.ai.ml import command

# configure job

job = command(
    code="./src",
    command="python train-model-mlflow.py --training_data ER_Triage_Dataset.csv",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
    display_name="ERTriage-train-mlflow",
    experiment_name="ERTriage-training", 
    tags={"model_type": "LogisticRegression"}
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.02 MBs): 100%|███████

Monitor your job at https://ml.azure.com/runs/hungry_feather_dyrhfvrdcc?wsid=/subscriptions/7784b486-c1df-4244-be68-cefd0413ea59/resourcegroups/rg-dp100-l44ab45990a0b486a80/workspaces/mlw-dp100-l44ab45990a0b486a80&tid=0c4563c3-3bc8-450f-8008-bc0edc7c121e


In the Studio, navigate to the **diabetes-train-mlflow** job to explore the overview of the command job you ran:

- Find the logged parameters in the **Overview** tab, under **Params**.
- Find the logged metrics in the **Metrics** tab.
- Find the logged artifacts in the **Images** tab (specifically for images), and in the **Outputs + logs** tab (all files).

## Autologging with MLflow

Instead of using custom logging, MLflow can also automatically log any parameters, metrics, and artifacts. Autologging with MLflow requires only one line of code.

Run the following cell to create the **train-model-autolog.py** script in the **src** folder. The script trains a classification model by using the **diabetes.csv** file in the same folder, which is passed as an argument. 

Review the code below to find that the script will import `mlflow` and enables autologging with the line: 

`mlflow.autolog()`

In [7]:
%%writefile $script_folder/train-model-autolog.py

# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def main(args):

    # enable autologging
    mlflow.autolog()

    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):

    print("Reading data...")

    df = pd.read_csv(path)

    # label encoding categorical columns
    le_arrival = LabelEncoder()
    le_symptom = LabelEncoder()
    le_target = LabelEncoder()

    df['Arrival_Mode'] = le_arrival.fit_transform(df['Arrival_Mode'])

    df['Primary_Symptom'] = le_symptom.fit_transform(
        df['Primary_Symptom']
    )

    df['Routing_Class'] = le_target.fit_transform(
        df['Routing_Class']
    )

    # normalization
    scaler = MinMaxScaler()

    df[['Age', 'Arrival_Mode', 'Heart_Rate',
        'Pain_Scale', 'Primary_Symptom']] = scaler.fit_transform(
        df[['Age', 'Arrival_Mode', 'Heart_Rate',
        'Pain_Scale', 'Primary_Symptom']]
    )

    return df

# function that splits the data
def split_data(df):

    print("Splitting data...")

    X, y = df[['Age','Arrival_Mode','Heart_Rate',
    'Pain_Scale','Primary_Symptom']].values, df[
    'Routing_Class'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=0
    )

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):

    mlflow.log_param("Regularization rate", reg_rate)

    print("Training model...")

    model = LogisticRegression(
        C=1/reg_rate,
        solver="liblinear"
    ).fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):

    # calculate accuracy
    y_hat = model.predict(X_test)

    acc = np.average(y_hat == y_test)

    print('Accuracy:', acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)

    auc = roc_auc_score(y_test, y_scores[:,1])

    print('AUC: ' + str(auc))

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(
        y_test,
        y_scores[:,1]
    )

    fig = plt.figure(figsize=(6, 4))

    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')

    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)

    plt.xlabel('False Positive Rate')

    plt.ylabel('True Positive Rate')

    plt.title('ROC Curve')

    plt.savefig("ROC-Curve.png")

def parse_args():

    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument(
        "--training_data",
        dest='training_data',
        type=str
    )

    parser.add_argument(
        "--reg_rate",
        dest='reg_rate',
        type=float,
        default=0.01
    )

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":

    # add space in logs
    print("\n\n")

    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)

    print("\n\n")

Writing src/train-model-autolog.py


Now, you can submit the script as a command job.

Run the cell below to train the model. 

In [8]:
from azure.ai.ml import command

# configure job

job = command(
    code="./src",
    command="python train-model-autolog.py --training_data ER_Triage_Dataset.csv",
    environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    compute="aml-cluster",
    display_name="ERTriage-train-autolog",
    experiment_name="ERTriage-training"
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Uploading src (0.03 MBs): 100%|██████████| 25794/25794 [00:00<00:00, 486091.28it/s]




Monitor your job at https://ml.azure.com/runs/honest_planet_c6ky0c646d?wsid=/subscriptions/7784b486-c1df-4244-be68-cefd0413ea59/resourcegroups/rg-dp100-l44ab45990a0b486a80/workspaces/mlw-dp100-l44ab45990a0b486a80&tid=0c4563c3-3bc8-450f-8008-bc0edc7c121e


In the Studio, navigate to the **diabetes-train-autolog** job to explore the overview of the command job you ran:

- Find the logged parameters in the **Overview** tab, under **Params**.
- Find the logged metrics in the **Metrics** tab.
- Find the logged artifacts in the **Images** tab (specifically for images), and in the **Outputs + logs** tab (all files, including the model files).

## Use MLflow to view and search for experiments

The Azure Machine Learning Studio is an easy-to-use UI to view and compare job runs. Alternatively, you can use MLflow to view experiment jobs. 

To list the jobs in the workspace, use the following command to list the experiments in the workspace:


In [9]:
import mlflow
experiments = mlflow.search_experiments()
for exp in experiments:
    print(exp.name)

auto-ml-class-dev
ERTriage-training


To retrieve a specific experiment, you can get it by its name:

In [10]:
experiment_name = "ERTriage-training"
exp = mlflow.get_experiment_by_name(experiment_name)
print(exp)

<Experiment: artifact_location='', creation_time=1778409245597, experiment_id='f0f97202-2d9a-48c7-b92e-d8147277a260', last_update_time=None, lifecycle_stage='active', name='ERTriage-training', tags={}>


Using an experiment name, you can retrieve all jobs of that experiment:

In [11]:
mlflow.search_runs(exp.experiment_id)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,metrics.AUC,metrics.training_precision_score,metrics.training_roc_auc_score,...,params.max_iter,params.l1_ratio,params.C,params.dual,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.rootRunId,tags.model_type,tags.estimator_name,tags.estimator_class
0,nifty_monkey_yjhzzhbnw1,f0f97202-2d9a-48c7-b92e-d8147277a260,FAILED,,2026-05-10 10:36:39.904000+00:00,2026-05-10 10:38:01.300000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,Thanshi,ERTriage-train-script,nifty_monkey_yjhzzhbnw1,None,None,None
1,patient_pig_7cn3kvlhnj,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:44:19.607000+00:00,2026-05-10 10:46:03.614000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,Thanshi,ERTriage-train-script,patient_pig_7cn3kvlhnj,None,None,None
2,hungry_feather_dyrhfvrdcc,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:55:03.205000+00:00,2026-05-10 10:57:05.597000+00:00,0.866667,0.92617,NaN,NaN,...,None,None,None,None,Thanshi,ERTriage-train-mlflow,hungry_feather_dyrhfvrdcc,LogisticRegression,None,None
3,honest_planet_c6ky0c646d,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:58:10.478000+00:00,2026-05-10 10:58:36.945000+00:00,NaN,NaN,0.901643,0.95472,...,100,None,100.0,False,Thanshi,ERTriage-train-autolog,honest_planet_c6ky0c646d,None,LogisticRegression,sklearn.linear_model._logistic.LogisticRegression


To more easily compare job runs and outputs, you can configure the search to order the results. For example, the following cell orders the results by `start_time`, and only shows a maximum of `2` results: 

In [12]:
mlflow.search_runs(exp.experiment_id, order_by=["start_time DESC"], max_results=2)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.training_roc_auc_score,metrics.training_recall_score,metrics.training_precision_score,metrics.training_f1_score,...,params.l1_ratio,params.Regularization rate,params.C,params.dual,tags.estimator_name,tags.mlflow.user,tags.mlflow.runName,tags.estimator_class,tags.mlflow.rootRunId,tags.model_type
0,honest_planet_c6ky0c646d,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:58:10.478000+00:00,2026-05-10 10:58:36.945000+00:00,0.95472,0.902857,0.901643,0.901931,...,None,0.01,100.0,False,LogisticRegression,Thanshi,ERTriage-train-autolog,sklearn.linear_model._logistic.LogisticRegression,honest_planet_c6ky0c646d,None
1,hungry_feather_dyrhfvrdcc,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:55:03.205000+00:00,2026-05-10 10:57:05.597000+00:00,NaN,NaN,NaN,NaN,...,None,0.01,None,None,None,Thanshi,ERTriage-train-mlflow,None,hungry_feather_dyrhfvrdcc,LogisticRegression


You can even create a query to filter the runs. Filter query strings are written with a simplified version of the SQL `WHERE` clause. 

To filter, you can use two classes of comparators:

- Numeric comparators (metrics): =, !=, >, >=, <, and <=.
- String comparators (params, tags, and attributes): = and !=.

Learn more about [how to track experiments with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow).

In [13]:
query = "metrics.AUC > 0.8 and tags.model_type = 'LogisticRegression'"
mlflow.search_runs(exp.experiment_id, filter_string=query)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.Accuracy,metrics.AUC,params.Regularization rate,tags.mlflow.runName,tags.model_type,tags.mlflow.rootRunId,tags.mlflow.user
0,hungry_feather_dyrhfvrdcc,f0f97202-2d9a-48c7-b92e-d8147277a260,FINISHED,,2026-05-10 10:55:03.205000+00:00,2026-05-10 10:57:05.597000+00:00,0.866667,0.92617,0.01,ERTriage-train-mlflow,LogisticRegression,hungry_feather_dyrhfvrdcc,Thanshi
